In [1]:
# -*- coding: utf-8 -*-
"""
UWB Single-Sided Ranging (SSR)
==============================

Reconstructs range from single-sided (poll-only) timestamps and shows how the
estimate is progressively improved:

    Baseline 1  no correction ....... raw to_s() timestamps, single global D/S fit
    Baseline 2  initial approach .... unwrapped timestamps + zero-step NTP fix,
                                       single global D/S fit
    Final fix   sliding window ...... unwrapped timestamps + typical-step NTP fix,
                                       local (windowed) D/S fit + outlier rejection

D = local clock drift (ppm), S = local clock offset (s), estimated by least
squares from the two-way exchange timestamps. The DS-TWR range reported by the
device is kept only as a reference for how close SSR gets.

Only single-sided-ranging code is retained in this file.
"""

import json

import numpy as np
import matplotlib.pyplot as plt


# ─────────────────────────────────────────────────────────────────────────────
# Configuration & physical constants
# ─────────────────────────────────────────────────────────────────────────────
IN_PATH        = "uwb_raw.json"   # input JSON dataset
OUT_PNG        = "ssr_analysis.png"            # output figure

GT             = 0.5                           # ground-truth range [m]
SPEED_OF_LIGHT = 299_702_547                   # [m/s] (in air)
DWT_TIME_UNITS = 1.0 / 499.2e6 / 128.0         # DW1000 tick period [s]
T_WRAP_DTU     = 2 ** 32                        # 32-bit counter wrap [ticks]
T_WRAP_S       = T_WRAP_DTU * DWT_TIME_UNITS   # wrap period [s] (~0.0672 s)

WINDOW         = 10                            # sliding-window size for local fit
OUTLIER_HALF_WIN = 15                          # half-window for outlier rejection
OUTLIER_K        = 3.0                         # MAD threshold for outlier rejection


# ─────────────────────────────────────────────────────────────────────────────
# Time helpers
# ─────────────────────────────────────────────────────────────────────────────
def to_s(dtu):
    """Convert a raw DW tick count to seconds (no wrap handling)."""
    return dtu * DWT_TIME_UNITS


def unwrap_to_s(raw_dtu, corr_wall_t, t0_wall):
    """
    Reconstruct absolute DWT time (seconds) from a raw 32-bit counter value.
    The wall clock is used only to identify which ~67 ms wrap cycle we are in.
    """
    n = round((corr_wall_t - t0_wall) / T_WRAP_S)
    return (raw_dtu + n * T_WRAP_DTU) * DWT_TIME_UNITS


# ─────────────────────────────────────────────────────────────────────────────
# Data loading & filtering
# ─────────────────────────────────────────────────────────────────────────────
def load_and_filter(path):
    """
    Load UWB records and keep only physically consistent two-way exchanges.

    An exchange is accepted when its source and destination timestamps are
    strictly ascending and the destination poll arrives after the source poll.

    Returns
        ranges          : list of accepted records
        reported_ranges : device-reported (DS-TWR) range for each accepted record
    """
    all_records = json.load(open(path, "r"))
    print(f"Loaded {len(all_records)} records")

    ranges, reported_ranges, n_filtered = [], [], 0
    for r in all_records:
        if r["type"] != "uwb":
            continue
        ascending_src = r["poll_src_tx"] < r["resp_src_rx"] < r["final_src_tx"]
        ascending_dst = r["poll_dst_rx"] < r["resp_dst_tx"] < r["final_dst_rx"]
        dst_greater   = to_s(r["poll_src_tx"]) + 5e-3 < to_s(r["poll_dst_rx"])

        if ascending_src and ascending_dst and dst_greater:
            ranges.append(r)
            reported_ranges.append(r["range"])
        else:
            n_filtered += 1

    print(f"Clean exchanges : {len(ranges)}")
    print(f"Filtered out    : {n_filtered}")
    return ranges, np.array(reported_ranges)


# ─────────────────────────────────────────────────────────────────────────────
# Wall-clock NTP correction
# ─────────────────────────────────────────────────────────────────────────────
def make_corrected_wall(ranges):
    """
    Final NTP fix: detect backward jumps in r['t'] and replace each with a
    forward step of the typical inter-exchange interval.

    Returns
        corrected : monotonically increasing wall times [s]
        ntp_jumps : set of sample indices where a jump was detected
    """
    raw_diffs = [ranges[i]["t"] - ranges[i - 1]["t"] for i in range(1, len(ranges))]
    typical   = np.median([d for d in raw_diffs if d > 0])

    corrected, ntp_jumps, offset = [ranges[0]["t"]], set(), 0.0
    for i in range(1, len(ranges)):
        dt = ranges[i]["t"] - ranges[i - 1]["t"]
        if dt < 0:
            offset = (corrected[-1] + typical) - ranges[i]["t"]
            ntp_jumps.add(i)
            print(f"  NTP jump at sample {i:3d}: {dt*1000:+.0f} ms  ->  "
                  f"corrected to +{typical*1000:.0f} ms")
        corrected.append(ranges[i]["t"] + offset)

    return corrected, ntp_jumps


def make_corrected_wall_v1(ranges):
    """
    Original (Baseline 2) NTP fix: a backward jump sets corrected[i] =
    corrected[i-1] (zero forward step instead of the typical interval).
    """
    corrected, offset = [ranges[0]["t"]], 0.0
    for i in range(1, len(ranges)):
        dt = ranges[i]["t"] - ranges[i - 1]["t"]
        if dt < 0:
            offset -= dt
        corrected.append(ranges[i]["t"] + offset)
    return corrected


# ─────────────────────────────────────────────────────────────────────────────
# SSR least-squares building blocks
# ─────────────────────────────────────────────────────────────────────────────
def _ssr_rows(ps, rs, fs, pd, rd, fd, tof_):
    """
    Linear-system rows for one exchange, modelling the destination clock as
    t_src = (1 + D) * t_dst + S. Unknowns are [D, S].
    """
    A = [[pd, 1], [rd, 1], [fd, 1]]
    b = [ps + tof_ - pd,
         rs - tof_ - rd,
         fs + tof_ - fd]
    return A, b


def fit_ssr_global(ranges, timestamps_of):
    """
    Fit a single global (D, S) over every exchange.

    timestamps_of(i, r) must return (ps, rs, fs, pd, rd, fd) in seconds.
    """
    tof_ = GT / SPEED_OF_LIGHT
    A, b = [], []
    for i, r in enumerate(ranges):
        rows_A, rows_b = _ssr_rows(*timestamps_of(i, r), tof_)
        A += rows_A
        b += rows_b
    x, *_ = np.linalg.lstsq(np.array(A), np.array(b), rcond=None)
    return x[0], x[1]                       # D, S


def ssr_error_global(ranges, D, S, poll_of):
    """
    Absolute SSR range error for a global (D, S) fit.
    poll_of(i, r) must return (ps, pd) in seconds.
    """
    err = []
    for i, r in enumerate(ranges):
        ps, pd  = poll_of(i, r)
        tof_est = (1 + D) * pd + S - ps
        err.append(abs(SPEED_OF_LIGHT * tof_est - GT))
    return np.array(err)


def compute_ssr_windowed(ranges, wall_corrected, t0, ntp_jumps, window=WINDOW):
    """
    Final SSR estimate. For each exchange i, estimate a *local* (D, S) from the
    nearest `window` exchanges (NTP-boundary samples excluded), then compute the
    SSR ToF from the poll timestamps only.

    Returns tofs, D_log, S_log (all length len(ranges); NaN where under-determined).
    """
    tof_  = GT / SPEED_OF_LIGHT
    tofs, D_log, S_log = [], [], []

    for i in range(len(ranges)):
        lo = max(0, i - window // 2)
        hi = min(len(ranges), lo + window)
        lo = max(0, hi - window)

        A, b = [], []
        for idx in range(lo, hi):
            if idx in ntp_jumps:
                continue
            rw, cw = ranges[idx], wall_corrected[idx]
            ts = (unwrap_to_s(rw["poll_src_tx"],  cw, t0),
                  unwrap_to_s(rw["resp_src_rx"],  cw, t0),
                  unwrap_to_s(rw["final_src_tx"], cw, t0),
                  unwrap_to_s(rw["poll_dst_rx"],  cw, t0),
                  unwrap_to_s(rw["resp_dst_tx"],  cw, t0),
                  unwrap_to_s(rw["final_dst_rx"], cw, t0))
            rows_A, rows_b = _ssr_rows(*ts, tof_)
            A += rows_A
            b += rows_b

        if len(A) < 6:
            tofs.append(np.nan); D_log.append(np.nan); S_log.append(np.nan)
            continue

        x, *_ = np.linalg.lstsq(np.array(A), np.array(b), rcond=None)
        D_l, S_l = x[0], x[1]
        D_log.append(D_l)
        S_log.append(S_l)

        ps_i = unwrap_to_s(ranges[i]["poll_src_tx"], wall_corrected[i], t0)
        pd_i = unwrap_to_s(ranges[i]["poll_dst_rx"], wall_corrected[i], t0)
        tofs.append((1 + D_l) * pd_i + S_l - ps_i)

    return np.array(tofs), np.array(D_log), np.array(S_log)


# ─────────────────────────────────────────────────────────────────────────────
# Outlier rejection
# ─────────────────────────────────────────────────────────────────────────────
def reject_outliers(tofs, half_win=OUTLIER_HALF_WIN, k=OUTLIER_K):
    """
    Flag any sample whose ToF deviates from its rolling median by > k * MAD,
    then replace flagged samples by linear interpolation from clean neighbours.

    Returns clean_tofs, mask.
    """
    n    = len(tofs)
    mask = np.zeros(n, dtype=bool)

    for i in range(n):
        lo, hi = max(0, i - half_win), min(n, i + half_win + 1)
        valid  = tofs[lo:hi][~np.isnan(tofs[lo:hi])]
        if len(valid) < 3:
            continue
        med = np.median(valid)
        mad = np.median(np.abs(valid - med))
        if mad > 0 and abs(tofs[i] - med) > k * mad:
            mask[i] = True

    valid_idx  = np.where(~mask & ~np.isnan(tofs))[0]
    clean_tofs = np.interp(np.arange(n), valid_idx, tofs[valid_idx])

    print(f"Outliers removed : {mask.sum()} samples")
    print(f"Indices          : {np.where(mask)[0].tolist()}")
    return clean_tofs, mask


# ─────────────────────────────────────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────────────────────────────────────
def plot_results(ranges, ntp_jumps,
                 ssr_err_b1, ssr_err_b2, ssr_err,
                 dstwr_reported_err, D_log, S_log,
                 tofs, clean_tofs, outlier_mask, out_png=OUT_PNG):
    samples = np.arange(len(ranges))
    fig, axes = plt.subplots(5, 1, figsize=(12, 22))

    # Plot 1: all error curves (log scale)
    ax = axes[0]
    ax.semilogy(samples, ssr_err_b1, color="red", linewidth=1.0, alpha=0.8,
                label=f"Baseline 1 - no correction        (mean={ssr_err_b1.mean():.0f} m)")
    ax.semilogy(samples, ssr_err_b2, color="darkorange", linewidth=1.0, alpha=0.8,
                label=f"Baseline 2 - initial approach      (mean={ssr_err_b2.mean():.0f} m)")
    ax.semilogy(samples, ssr_err, color="steelblue", linewidth=1.2,
                label=f"Final fix  - sliding window + NTP  (mean={ssr_err.mean():.2f} m)")
    ax.semilogy(samples, dstwr_reported_err, color="green", linewidth=1.0, alpha=0.6,
                linestyle="--",
                label=f"DS-TWR reported (reference)        (mean={dstwr_reported_err.mean():.3f} m)")
    ax.set_title("SSR Ranging Error - All Methods (log scale)")
    ax.set_xlabel("Sample"); ax.set_ylabel("Absolute Error (m)  [log]")
    ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.4)

    # Plot 2: final SSR error with statistics
    ax = axes[1]
    ax.plot(samples, ssr_err, color="steelblue", linewidth=1.2, label="SSR error")
    ax.axhline(ssr_err.mean(), color="red", linestyle="--", linewidth=1.2,
               label=f"Mean  = {ssr_err.mean():.3f} m")
    ax.axhline(ssr_err.mean() + ssr_err.std(), color="orange", linestyle=":", linewidth=1.2,
               label=f"Mean+1sigma = {ssr_err.mean()+ssr_err.std():.3f} m")
    ax.axhline(ssr_err.max(), color="black", linestyle=":", linewidth=1.0,
               label=f"Max   = {ssr_err.max():.3f} m")
    ax.fill_between(samples, 0, ssr_err, alpha=0.15, color="steelblue")
    ax.scatter(np.where(outlier_mask)[0], ssr_err[outlier_mask],
               color="red", zorder=5, s=20, label="Interpolated outliers")
    ax.set_title("Final SSR Ranging Error (m)")
    ax.set_xlabel("Sample"); ax.set_ylabel("Absolute Error (m)")
    ax.legend(fontsize=9); ax.grid(True)

    # Plot 3: local D (clock drift)
    ax = axes[2]
    ax.plot(samples, np.array(D_log) * 1e6, color="darkorange")
    for j in sorted(ntp_jumps):
        ax.axvline(j, color="red", linestyle="--", alpha=0.5, linewidth=0.9)
    ax.set_title("Local Clock Drift D (ppm)  -  red dashes = NTP jump locations")
    ax.set_xlabel("Sample"); ax.set_ylabel("D (ppm)"); ax.grid(True)

    # Plot 4: local S (clock offset)
    ax = axes[3]
    ax.plot(samples, np.array(S_log) * 1e6, color="purple")
    for j in sorted(ntp_jumps):
        ax.axvline(j, color="red", linestyle="--", alpha=0.5, linewidth=0.9)
    ax.set_title("Local Clock Offset S (us)  -  should be nearly flat after fix")
    ax.set_xlabel("Sample"); ax.set_ylabel("S (us)"); ax.grid(True)

    # Plot 5: raw vs clean range
    ax = axes[4]
    ax.plot(samples, tofs * SPEED_OF_LIGHT * 100,
            label="Before outlier rejection", color="gray", alpha=0.6)
    ax.plot(samples, clean_tofs * SPEED_OF_LIGHT * 100,
            label="After outlier rejection", color="steelblue")
    ax.axhline(GT * 100, color="black", linestyle="--", linewidth=1,
               label=f"Ground truth ({GT} m)")
    ax.set_title("SSR Estimated Range (cm)  -  before and after outlier rejection")
    ax.set_xlabel("Sample"); ax.set_ylabel("Range (cm)")
    ax.legend(fontsize=9); ax.grid(True)

    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline
# ─────────────────────────────────────────────────────────────────────────────
def main():
    # 1. Load & filter -------------------------------------------------------
    ranges, reported_ranges = load_and_filter(IN_PATH)
    dstwr_reported_err = np.abs(reported_ranges - GT)

    # 2. Wall-clock NTP correction (final version) --------------------------
    print("Correcting wall clock for NTP jumps:")
    wall_corrected, ntp_jumps = make_corrected_wall(ranges)
    t0 = wall_corrected[0]
    interval = np.median([wall_corrected[i] - wall_corrected[i - 1]
                          for i in range(1, len(wall_corrected))])
    print(f"\nNTP jump indices : {sorted(ntp_jumps)}")
    print(f"Typical interval : {interval*1000:.1f} ms")

    # 3. Windowed SSR (final estimate) --------------------------------------
    tofs, D_log, S_log = compute_ssr_windowed(ranges, wall_corrected, t0, ntp_jumps)
    print(f"D range: {np.nanmin(D_log)*1e6:.4f} to {np.nanmax(D_log)*1e6:.4f} ppm")
    print(f"S range: {np.nanmin(S_log):.6f} to {np.nanmax(S_log):.6f} s")

    # 4. Baseline 1 - no correction (raw to_s timestamps, global fit) -------
    def ts_raw(i, r):
        return (to_s(r["poll_src_tx"]), to_s(r["resp_src_rx"]), to_s(r["final_src_tx"]),
                to_s(r["poll_dst_rx"]), to_s(r["resp_dst_tx"]), to_s(r["final_dst_rx"]))

    D_b1, S_b1 = fit_ssr_global(ranges, ts_raw)
    ssr_err_b1 = ssr_error_global(
        ranges, D_b1, S_b1,
        poll_of=lambda i, r: (to_s(r["poll_src_tx"]), to_s(r["poll_dst_rx"])))
    print(f"Baseline 1 (no correction)    - mean: {ssr_err_b1.mean():.1f} m   "
          f"max: {ssr_err_b1.max():.1f} m")

    # 5. Baseline 2 - initial approach (unwrapped + zero-step NTP, global fit)
    wall_v1 = make_corrected_wall_v1(ranges)
    t0_b2   = wall_v1[0]

    def ts_v1(i, r):
        cw = wall_v1[i]
        return (unwrap_to_s(r["poll_src_tx"],  cw, t0_b2),
                unwrap_to_s(r["resp_src_rx"],  cw, t0_b2),
                unwrap_to_s(r["final_src_tx"], cw, t0_b2),
                unwrap_to_s(r["poll_dst_rx"],  cw, t0_b2),
                unwrap_to_s(r["resp_dst_tx"],  cw, t0_b2),
                unwrap_to_s(r["final_dst_rx"], cw, t0_b2))

    D_b2, S_b2 = fit_ssr_global(ranges, ts_v1)
    ssr_err_b2 = ssr_error_global(
        ranges, D_b2, S_b2,
        poll_of=lambda i, r: (unwrap_to_s(r["poll_src_tx"], wall_v1[i], t0_b2),
                              unwrap_to_s(r["poll_dst_rx"], wall_v1[i], t0_b2)))
    print(f"Baseline 2 (initial approach) - mean: {ssr_err_b2.mean():.1f} m   "
          f"max: {ssr_err_b2.max():.1f} m")

    # 6. Outlier rejection on the final SSR ToFs ----------------------------
    clean_tofs, outlier_mask = reject_outliers(tofs)
    ssr_ranges = SPEED_OF_LIGHT * clean_tofs
    ssr_err    = np.abs(ssr_ranges - GT)
    print(f"\nSSR - mean: {ssr_err.mean():.3f} m   max: {ssr_err.max():.3f} m   "
          f"std: {ssr_err.std():.3f} m")

    # 7. Plots & summary -----------------------------------------------------
    plot_results(ranges, ntp_jumps, ssr_err_b1, ssr_err_b2, ssr_err,
                 dstwr_reported_err, D_log, S_log, tofs, clean_tofs, outlier_mask)

    print("\n=== Final Summary ===")
    print(f"Baseline 1 - no correction    : mean={ssr_err_b1.mean():.1f} m   "
          f"max={ssr_err_b1.max():.1f} m")
    print(f"Baseline 2 - initial approach : mean={ssr_err_b2.mean():.1f} m   "
          f"max={ssr_err_b2.max():.1f} m")
    print(f"Final fix  - full pipeline    : mean={ssr_err.mean():.3f} m   "
          f"max={ssr_err.max():.3f} m")
    print(f"DS-TWR reported (reference)   : mean={dstwr_reported_err.mean():.3f} m   "
          f"max={dstwr_reported_err.max():.3f} m")
    print(f"NTP jumps detected : {sorted(ntp_jumps)}")
    print(f"Outliers removed   : {outlier_mask.sum()}")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: '/content/uwb_raw (1).json'